md

In [ ]:
# Registered constants, read from the pre-registration at build time.
SEEDS_HERE = [1401, 1402, 1403]
STOP_SPREAD_HERE = 0.15
print('registered seeds:', SEEDS_HERE, '| stopping threshold:', STOP_SPREAD_HERE)

In [ ]:
import glob, json, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))
    assert hits, 'attach the kalia-code-dev dataset'
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach a prep output as a kernel source'
DATA = os.path.dirname(data[0])
print('code:', work)
print('data:', DATA)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('adaptive_every supported:', 'adaptive_every' in open('train.py').read())

In [ ]:
# Per-source shards. mix_bins.py pre-blends for v0.2.0; this arm needs them
# separate. If the per-source shards are not attached, that is a dataset
# problem to solve on Kaggle CPU, not something to fake locally.
# Names must match what kalia-publish-sources uploads. They did not: the
# publisher wrote corpus/val_forget.bin and the notebook looked for
# checkpoints/probe_val.bin, so Step 4 would have failed on the missing
# file rather than on anything about the mixture.
SRC = {'fineweb': 'corpus/fineweb.bin', 'tinystories': 'corpus/tinystories.bin',
       'cosmopedia': 'corpus/cosmopedia.bin', 'python': 'corpus/python.bin'}
found = {}
for name, fn in SRC.items():
    hits = glob.glob(f'/kaggle/input/**/{fn}', recursive=True)
    print(f'{name:14s} {hits[0] if hits else "MISSING"}')
    if hits:
        found[name] = hits[0]
assert len(found) == 4, (
    f'need all 4 per-source shards, have {sorted(found)}. mix_bins.py\n'
    'pre-blends the sources into one train.bin and discards the originals,\n'
    'so these must be published by kalia-publish-sources first. Without\n'
    'them there is no runtime mixture to reweight, and the adaptive arm\n'
    'would silently be the static one.')
STATIC = {'fineweb': 0.60, 'tinystories': 0.20, 'cosmopedia': 0.15, 'python': 0.05}
names = list(STATIC)
print('\nstatic weights:', STATIC)

In [ ]:
def run(arm, adaptive_every, tag):
    out = f'/kaggle/working/out/{arm}'
    cmd = [sys.executable, 'train.py', '--config', 'configs/kalia-kautilya.yaml',
           '--data-dir', DATA, '--out-dir', out, '--max-steps', '1000',
           '--resume', '--hub-repo', 'kalia-lm/kalia-v020',
           '--seed', '1401',
           '--sources', *names,
           '--source-bins', *[found[n] for n in names],
           '--source-weights', json.dumps(STATIC)]
    if adaptive_every:
        cmd += []  # cadence comes from the config; see below
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f'--- {arm} (rc={r.returncode}) ---')
    print(r.stdout[-2500:])
    if r.returncode:
        print(r.stderr[-2500:])
    return r

import yaml
cfg = yaml.safe_load(open('configs/kalia-kautilya.yaml'))
print('config adaptive_every:', cfg['train']['adaptive_every'])
print('max_steps:', cfg['train']['max_steps'], '| seed:', cfg['train']['seed'])

In [ ]:
# The config is written to disk per arm so the two runs differ in exactly one
# key. Editing the loaded dict and hoping train.py re-reads it would make the
# arms differ by whatever the loader did.
import copy
def write_cfg(path, adaptive_every):
    c = copy.deepcopy(cfg)
    c['train']['adaptive_every'] = adaptive_every
    with open(path, 'w') as fh:
        yaml.safe_dump(c, fh)
    return path

static_cfg = write_cfg('/kaggle/working/kautilya-static.yaml', 0)
adapt_cfg = write_cfg('/kaggle/working/kautilya-adaptive.yaml', 100)
print('static   ->', open(static_cfg).read().split('adaptive_every:')[1].split()[0])
print('adaptive ->', open(adapt_cfg).read().split('adaptive_every:')[1].split()[0])
other = yaml.safe_load(open(adapt_cfg))['train']
print('differing keys:', [k for k in cfg['train'] if cfg['train'][k] != other[k]])